# A/B Testing: Evaluating a New Search-Ranking System

# 1. Business Problem & Objective

## Business Problem

An online booking platform is considering implementing a new search-ranking system to improve the user experience and encourage more bookings. However, before rolling it out to all users, the company needs to determine whether the new system improves conversion without negatively affecting the booking process.

Implementing a change without sufficient evidence could lead to a poor user experience or fail to deliver the expected business benefits. An A/B test allows the company to compare the existing search-ranking system (control group) with the new system (variant group) using observed user behaviour.

## Objective

The objective of this analysis is to evaluate the performance of the new search-ranking system and determine whether the evidence supports a full rollout.

The experiment will assess two metrics:

* **Primary metric — Conversion rate:** Measures the proportion of sessions that result in a booking. The goal is to determine whether the new system produces a statistically significant improvement in conversion.
* **Guardrail metric — Time to booking:** Measures the time taken to make a booking. This metric helps assess whether the new system is associated with a change in the booking process that could indicate a potential usability concern.

## Hypotheses and Decision Criteria

The analysis uses a significance level of **α = 0.10**.

For the primary metric,
- Null hypothesis states that the control and variant groups have equal conversion rates
- Alternative hypothesis states that their conversion rates differ.

For the guardrail metric, 
- Null hypothesis states that the groups have equal mean times to booking.
- Alternative hypothesis states that their mean times differ.

The analysis will also check for sample ratio mismatch (SRM) to assess whether the observed allocation between the control and variant groups is consistent with the expected experiment allocation.

The final recommendation will consider statistical significance, the direction and magnitude of the observed effects, and the experiment's rollout criteria.

# 2. Dataset Description 

## `sessions_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `session_id` | `string` | Unique session identifier (unique for each row) |
| `user_id` | `string` | Unique user identifier (non logged-in users have missing user_id values; each user can have multiple sessions) |
| `session_start_timestamp` | `string` | When a session started |
| `booking_timestamp` | `string` | When a booking was made (missing if no booking was made during a session) |
| `time_to_booking` | `float` | time from start of the session to booking, in minutes (missing if no booking was made during a session) |
| `conversion` | `integer` | _New column to create:_ did session end up with a booking (0 if booking_timestamp or time_to_booking is Null, otherwise 1) |

<br>

## `users_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `user_id` | `string` | Unique user identifier (only logged-in users in this table) |
| `experiment_group` | `string` | control / variant split for the experiment (expected to be equal 50/50) |

<br>

# 3. Data Loading & Initial Exploration

In [11]:
import pandas as pd
from scipy.stats import chisquare
from pingouin import ttest
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [12]:
sessions = pd.read_csv('sessions_data.csv')
users = pd.read_csv('users_data.csv')

In [13]:
sessions.info()
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 16981 entries, 0 to 16980
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   session_id               16981 non-null  str    
 1   user_id                  15283 non-null  str    
 2   session_start_timestamp  16981 non-null  str    
 3   booking_timestamp        2844 non-null   str    
 4   time_to_booking          2844 non-null   float64
dtypes: float64(1), str(4)
memory usage: 663.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id           10000 non-null  str  
 1   experiment_group  10000 non-null  str  
dtypes: str(2)
memory usage: 156.4 KB


In [14]:
print("Sessions Dataset")
print(sessions.isnull().sum())
print("-------------------------")
print("Users Dataset")
print(users.isnull().sum())

Sessions Dataset
session_id                     0
user_id                     1698
session_start_timestamp        0
booking_timestamp          14137
time_to_booking            14137
dtype: int64
-------------------------
Users Dataset
user_id             0
experiment_group    0
dtype: int64


In [15]:
sessions.sample(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking
4387,evPDOhm971fqG2Md,1Odxt2uAdMakKJKq,2025-01-17 06:46:18.213140249,2025-01-17 07:02:14.868614709,15.944258
7745,iFTsQaUdieIW8PsY,xIEkLU7SlNaWXp04,2025-01-08 23:58:10.773667097,NaN,NaN
5708,yLkXFZDSZG2bEvvO,wnBn6ftGWHYLcwJ2,2025-01-08 08:55:04.143507481,NaN,NaN
14168,Hx6Mz39ADKB7eaPK,ZL3oDES9zlvofN2N,2025-01-14 14:46:48.115243912,NaN,NaN
7346,ILftpsxPpteXrCYo,cdh0qI7JsTrQ2fIr,2025-01-29 05:06:34.712641239,NaN,NaN


In [16]:
users.sample(5)

,user_id,experiment_group
4813,LiKjcn4Lm1BC96OD,control
4712,SkvhME2iBjNJJ8NW,variant
6329,DjKflPKvAdD8djRL,control
3313,WKigCebCjpc12pmU,control
6403,TwwvIuXVTrEMBSmF,variant


In [17]:
users['experiment_group'].value_counts()

experiment_group
variant    5009
control    4991
Name: count, dtype: int64

### Missing-Value Analysis

The sessions dataset contains 1,698 missing values in `user_id` (approximately 10% of all sessions). According to the dataset documentation, these correspond to non-logged-in users who cannot be directly linked to the user-level experiment assignments.

Both `booking_timestamp` and `time_to_booking` contain 14,137 missing values (approximately 83.3% of sessions). This is consistent with the documented structure of the dataset, in which these fields are missing when a session does not result in a booking.

The users dataset contains no missing values in either `user_id` or `experiment_group`.

The missing booking-related values will be retained because they are necessary for distinguishing sessions with bookings from sessions without bookings. Sessions without a `user_id` will require special consideration when joining the datasets and conducting the experiment analysis.


# 4. Data Preparation

In [18]:
# Merge user and sessions dataset
sessions_x_users= sessions.merge(users,how= "inner", on="user_id")

In [19]:
sessions_x_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 15283 entries, 0 to 15282
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   session_id               15283 non-null  str    
 1   user_id                  15283 non-null  str    
 2   session_start_timestamp  15283 non-null  str    
 3   booking_timestamp        2607 non-null   str    
 4   time_to_booking          2607 non-null   float64
 5   experiment_group         15283 non-null  str    
dtypes: float64(1), str(5)
memory usage: 716.5 KB


In [20]:
# Create conversion column
sessions_x_users['conversion']= sessions_x_users['booking_timestamp'].isna()
sessions_x_users['conversion']= sessions_x_users['conversion'].map({True:0, False:1})

In [21]:
sessions_x_users.sample(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking,experiment_group,conversion
5134,1aRLFfra6g8vDYd5,9GRmJevlERyPtTgu,2025-01-10 13:08:37.846640110,NaN,NaN,control,0
3307,pjNTa2WGA8qL9x2N,iOhhXBYYItj65m2I,2025-01-30 16:24:19.446599960,NaN,NaN,control,0
5995,DVwGd51jWd3OD3Im,weSTAbQgD8tumzIb,2025-01-05 06:25:14.329483271,2025-01-05 06:34:15.405574825,9.017935,control,1
12036,oNnA9Gj37EXhJN9N,l6TyOBFz9BSKnPjZ,2025-01-23 03:26:32.391498327,NaN,NaN,variant,0
936,oWo0ToWrDqC8OujE,GlxHn2StSmFtvpIt,2025-01-20 20:15:22.009573936,NaN,NaN,variant,0


In [22]:
sessions_x_users['conversion'].value_counts()

conversion
0    12676
1     2607
Name: count, dtype: int64

### Data Preparation Summary

The sessions and users datasets were combined using an inner join on `user_id`, retaining 15,283 sessions with matching user records and excluding 1,698 sessions with missing user IDs.

A binary `conversion` variable was created using `booking_timestamp` to indicate whether a session resulted in a recorded booking. Sessions with a booking timestamp were assigned `1`, while sessions without one were assigned `0`.

The resulting dataset contained 15,283 sessions: 2,607 conversions and 12,676 non-conversions, giving an overall conversion rate of approximately 17.06%.

The prepared dataset was used for the subsequent experiment analysis, including the sample ratio mismatch check, comparison of conversion rates, and evaluation of time to booking.


# 5. Sample Ratio Mismatch (SRM) Check

In [23]:
confidence_level = 0.90  # Set the pre-defined confidence level (90%)
alpha = 1 - confidence_level  # Significance level for hypothesis tests

In [24]:
# Count the number of users in each group
control_users = users[users['experiment_group'] == 'control'].shape[0]
variant_users = users[users['experiment_group'] == 'variant'].shape[0]

total_users = control_users + variant_users

print("Control Users:", control_users)
print("Variant Users:", variant_users)

observed = [control_users, variant_users]
expected = [total_users / 2, total_users / 2]

chi = chisquare(observed, f_exp=expected)
print(chi)

srm_chi2_pval= chi.pvalue
print("p-value: ",srm_chi2_pval)
if chi.pvalue < 0.1:
    print("SRM may be present")
else:
    print("SRM may not be present")

Control Users: 4991
Variant Users: 5009
Power_divergenceResult(statistic=np.float64(0.0324), pvalue=np.float64(0.8571525681981985))
p-value:  0.8571525681981985
SRM may not be present


# 6. Primary Metric: Conversion Rate
 

In [31]:
#Effect Analysis 
n_A = sessions_x_users[sessions_x_users['experiment_group']=='control'].shape[0]
n_B = sessions_x_users[sessions_x_users['experiment_group']=='variant'].shape[0]

conv_A = sessions_x_users[sessions_x_users['experiment_group']=='control']['conversion'].sum()
conv_B = sessions_x_users[sessions_x_users['experiment_group']=='variant']['conversion'].sum()

print('Group A sessions:',n_A)
print('Group B sessions:',n_B)

conv_AB= [conv_A, conv_B]
n_AB= [n_A,n_B]

z_stat, pvalue = proportions_ztest(conv_AB, n_AB)
pval_primary= pvalue
(A_lo90,B_lo90) , (A_up90, B_up90) = proportion_confint(conv_AB, nobs= n_AB, alpha=alpha)

print(f"pvalue= {pvalue}")
print(f"Group A CI ({A_lo90}, {A_up90})")
print(f"Group B CI ({B_lo90}, {B_up90})")

Group A sessions: 7630
Group B sessions: 7653
pvalue= 0.00019761608793587393
Group A CI (0.15234972060916963, 0.16612996484299292)
Group B CI (0.17463639701606742, 0.18914251321521441)


In [26]:
# Effect Size
mean_B= sessions_x_users[sessions_x_users['experiment_group']=='variant']['conversion'].mean()
print("Mean Variant:",mean_B)
mean_A= sessions_x_users[sessions_x_users['experiment_group']=='control']['conversion'].mean()
print("Mean Control:",mean_A)
effect_size_primary= (mean_B-mean_A)/mean_A
print("Effect size: ",effect_size_primary)

Mean Variant: 0.18188945511564092
Mean Control: 0.15923984272608127
Effect size:  0.14223583747517704


### Summary

The variant achieved a conversion rate of approximately 18.19%, compared with 15.92% for the control group. The difference was statistically significant (p ≈ 0.000198), with an absolute uplift of 2.27 percentage points and a relative improvement of approximately 14.2%. These results suggest that the new ranking system may improve booking conversion, although the final rollout decision also depends on the guardrail metric and potential business implications.


# 7. Guardrail Metric: Time to Booking

In [27]:
#Effect Analysis
test_stat = ttest(
    sessions_x_users.loc[sessions_x_users['experiment_group'] == 'control', 'time_to_booking'],
    sessions_x_users.loc[sessions_x_users['experiment_group'] == 'variant', 'time_to_booking'],
    alternative="two-sided", paired=False
)

print(test_stat)

               T          dof alternative   p_val           CI95   cohen_d  \
T_test  0.618198  2575.103104   two-sided  0.5365  [-0.26, 0.49]  0.024224   

           power   BF10  
T_test  0.094593  0.053  


In [29]:
pval_guardrail = round(test_stat['p_val'].iloc[0], 4)
print(pval_guardrail)

0.5365


In [30]:
# Effect Size
mean_B= sessions_x_users[sessions_x_users['experiment_group']=='variant']['time_to_booking'].mean()
print("Mean Variant:",mean_B)
mean_A= sessions_x_users[sessions_x_users['experiment_group']=='control']['time_to_booking'].mean()
print("Mean Control:",mean_A)
effect_size_guardrail= (mean_B-mean_A)/mean_A
print("Effect size: ",effect_size_guardrail)

Mean Variant: 14.894028712999187
Mean Control: 15.012403966770629
Effect size:  -0.007885163098026175


### Summary

The independent two-sample t-test found no statistically significant difference in mean time to booking between the control and variant groups (p = 0.5365, α = 0.10). Mean booking time decreased from approximately 15.01 minutes to 14.89 minutes, a relative reduction of about 0.79%. Cohen's d was approximately 0.024 in magnitude, indicating a negligible standardized effect. These results provide no clear evidence of a change in booking time among converted sessions, but do not establish that the overall booking experience is unaffected.


# 8. Final Decision

The variant achieved a statistically significant improvement in conversion, with a relative uplift of approximately 14.2%. The guardrail analysis found no statistically significant difference in mean time to booking, and the observed relative change was a small reduction of approximately 0.79%.

Based on these results, a cautious rollout is recommended, accompanied by continued monitoring of conversion and the booking experience. Although the results are promising, the guardrail analysis only covers sessions that resulted in bookings and does not establish that all aspects of the customer experience remain unaffected.

